# Notebook 3: Transformación y Validación de Calidad
**Autor:** Leonardo Figueroa, Miguel Flores y Steven Villegas

**Pipeline ETL - NYC Taxi Trip Records**

**Objetivo:** Procesar cuarentena, transformar Bronze a Silver con campos derivados y validar calidad.


## Configuración Inicial


In [1]:
import sys, os, json, yaml, logging
from datetime import datetime
from pathlib import Path

# --- MEMORIA: fijar driver/executor ANTES de importar pyspark ---
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
os.environ['PYSPARK_SUBMIT_ARGS'] = '--driver-memory 4g --executor-memory 4g pyspark-shell'
# --------------------------------------------------------------

sys.path.append(os.path.abspath('..'))
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger('etl')
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

# Cargar config de spark desde YAML (shuffle.partitions, adaptive, etc.)
_config_path = os.path.join(os.path.abspath('..'), "config", "etl_config.yaml")
with open(_config_path, "r", encoding="utf-8") as _f:
    _cfg = yaml.safe_load(_f)
_spark_cfg = _cfg.get("spark", {})

builder = SparkSession.builder \
    .appName("NYC_Taxi_ETL_03") \
    .master(_spark_cfg.get("master", "local[*]"))

for k, v in _spark_cfg.get("config", {}).items():
    builder = builder.config(k, v)

spark = builder.getOrCreate()
logger.info(f"Spark memory: driver={spark.conf.get('spark.driver.memory', 'default')} "
            f"executor={spark.conf.get('spark.executor.memory', 'default')}")

from src.config_loader import *
from src.utils import generate_process_id, classify_error
from src.transformations import transform_to_silver
from src.quality_rules import validate_quality
PROCESS_ID = generate_process_id()


2026-06-18 17:50:09,770 - etl - INFO - Spark memory: driver=4g executor=default


---
## FASE 3: Cuarentena de Archivos Dañados
**Objetivo:** Identificar archivos que no pudieron leerse y clasificarlos por tipo de error.


In [2]:
inventory_path = os.path.join(METADATA_PATH, "audit_file_inventory")
quarantine_records = []
if os.path.exists(inventory_path):
    df_inv = spark.read.parquet(inventory_path)
    inventory = [row.asDict() for row in df_inv.collect()]
else:
    inventory = []
for item in inventory:
    if item["read_status"] != "SUCCESS":
        err_msg = item.get("error_message", "")
        category = classify_error(err_msg)
        quarantine_records.append({
            "process_id": PROCESS_ID, "file_name": item["file_name"],
            "service_type": item["service_type"], "rejection_category": category,
            "error_message": err_msg[:500],
            "stage": "phase3_ingestion",
            "recommended_action": "Redownload from source" if category.startswith("NOT") else "Attempt schema recovery",
            "quarantined_at": datetime.now().isoformat()
        })
        logger.warning(f"Quarantine: {item['file_name']} -> {category}")
if quarantine_records:
    df_q = spark.createDataFrame(quarantine_records)
    df_q.write.mode("append").parquet(os.path.join(QUARANTINE_PATH, "quarantine_log"))
    df_q_pd = df_q.toPandas()
    print("\nRegistros en cuarentena (desde inventario FASE 1):")
    display(df_q_pd)

# También leer registros previos en quarantine_log (FASE 1b bad_parquet)
qlog_path = os.path.join(QUARANTINE_PATH, "quarantine_log")
if os.path.exists(qlog_path):
    df_qlog = spark.read.parquet(qlog_path)
    qlog_count = df_qlog.count()
    if qlog_count > 0:
        print(f"\nTotal registros en quarantine_log (incluye FASE 1b bad_parquet): {qlog_count}")
        df_qlog.show(truncate=False)
    else:
        logger.info("No files in quarantine - all files readable")
else:
    logger.info("No files in quarantine - all files readable")



Total registros en quarantine_log (incluye FASE 1b bad_parquet): 3
+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------+----------+--------------------------+-----------------------+------------------------+------------+----------------+
|error_message                                                                                                                                                                                                                                                                            

---
## FASE 4: Transformación de Datos (Bronze → Silver)
**Objetivo:** Limpiar, calcular campos derivados y detectar registros sospechosos.


In [3]:
silver_paths = transform_to_silver(spark, SERVICES, YEARS)
logger.info(f"Silver layer: {len(silver_paths)} partitions created")
print(f"\nSilver partitions written: {len(silver_paths)}")


2026-06-17 20:28:16,964 - src.transformations - INFO - Silver: yellow_tripdata_2024-01.parquet - valid:2864053 susp:100571 dups:0
2026-06-17 20:28:33,879 - src.transformations - INFO - Silver: yellow_tripdata_2024-02.parquet - valid:2896203 susp:111322 dups:1
2026-06-17 20:28:50,996 - src.transformations - INFO - Silver: yellow_tripdata_2024-03.parquet - valid:3433987 susp:148640 dups:1
2026-06-17 20:29:08,621 - src.transformations - INFO - Silver: yellow_tripdata_2024-04.parquet - valid:3408460 susp:105829 dups:0
2026-06-17 20:29:25,851 - src.transformations - INFO - Silver: yellow_tripdata_2024-05.parquet - valid:3611767 susp:112066 dups:0
2026-06-17 20:29:42,544 - src.transformations - INFO - Silver: yellow_tripdata_2024-06.parquet - valid:3422383 susp:116810 dups:0
2026-06-17 20:29:57,880 - src.transformations - INFO - Silver: yellow_tripdata_2024-07.parquet - valid:2968472 susp:108430 dups:1
2026-06-17 20:30:13,630 - src.transformations - INFO - Silver: yellow_tripdata_2024-08.par


Silver partitions written: 84


---
## FASE 5: Validación de Calidad
**Objetivo:** Medir la calidad del dataset aplicando 9 reglas de negocio sobre registros sospechosos.


In [4]:
quality_metrics, quality_rejected = validate_quality(spark, SERVICES, YEARS, PROCESS_ID)
if quality_metrics:
    df_qm = spark.createDataFrame(quality_metrics)
    qm_path = os.path.join(METADATA_PATH, "quality_metrics_summary")
    df_qm.write.mode("overwrite").parquet(qm_path)
    df_qm_pd = df_qm.toPandas()
    print("\nMétricas de calidad por servicio/mes:")
    display(df_qm_pd)
if quality_rejected:
    df_qr = spark.createDataFrame(quality_rejected)
    qr_path = os.path.join(METADATA_PATH, "quality_rejected_records")
    df_qr.write.mode("overwrite").parquet(qr_path)
    logger.info(f"Quality rejected: {len(quality_rejected)} records")
else:
    logger.info("No quality rejected records")



Métricas de calidad por servicio/mes:


,duplicate_records,month,null_critical_records,process_id,processed_at,quality_percentage,rejected_records,service_type,suspicious_records,total_records,valid_records,year
0,0,1,0,055285cd25a5,2026-06-17T22:21:52.967205,96.61,100571,yellow,100571,2964624,2864053,2024
1,0,2,0,055285cd25a5,2026-06-17T22:21:57.932013,96.30,111322,yellow,111322,3007525,2896203,2024
2,0,3,0,055285cd25a5,2026-06-17T22:22:02.008416,95.85,148640,yellow,148640,3582627,3433987,2024
3,0,4,0,055285cd25a5,2026-06-17T22:22:05.869039,96.99,105829,yellow,105829,3514289,3408460,2024
4,0,5,0,055285cd25a5,2026-06-17T22:22:09.658662,96.99,112066,yellow,112066,3723833,3611767,2024
...,...,...,...,...,...,...,...,...,...,...,...,...
79,0,12,44216722,055285cd25a5,2026-06-17T22:33:40.199566,99.94,12597,fhvhv,12597,22108361,22095764,2025
80,0,1,41880612,055285cd25a5,2026-06-17T22:33:53.285927,99.82,37563,fhvhv,37563,20940306,20902743,2026
81,0,2,39751220,055285cd25a5,2026-06-17T22:34:08.333138,99.94,10962,fhvhv,10962,19875610,19864648,2026
82,0,3,44116576,055285cd25a5,2026-06-17T22:35:02.357887,99.94,12866,fhvhv,12866,22058288,22045422,2026


2026-06-17 22:35:45,350 - etl - INFO - Quality rejected: 2593 records


---
## Resumen de Transformación y Validación
Datos transformados a Silver en data/silver/. Métricas de calidad guardadas en data/audit/.
